# 04 — Train Models

**Goal:** Train 7 ML models on each of the 3 poverty thresholds (21 models total).  
**Reads:** `data/processed/train.csv`, `data/processed/test.csv`  
**Outputs:** 21 `.pkl` files in `models/`

Models: XGBoost (CPU), XGBoost (GPU), LightGBM, Random Forest, Ridge, MLP, GAM  
Targets: `poverty_3`, `poverty_8_30`, `poverty_10`

## Imports and paths

In [1]:
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from pygam import LinearGAM, s

# XGBoost and LightGBM depend on OpenMP (libomp).
# Wrap imports so the notebook doesn't crash if the runtime library is missing.
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except (ImportError, OSError) as e:
    HAS_XGBOOST = False
    print(f"⚠ XGBoost unavailable: {e}")

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except (ImportError, OSError) as e:
    HAS_LIGHTGBM = False
    print(f"⚠ LightGBM unavailable: {e}")

import warnings
warnings.filterwarnings('ignore')

DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
TARGETS = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)

print(f"Features: {FEATURES}")
print(f"Targets:  {TARGETS}")

Features: ['gdp_per_capita', 'hdi', 'control_of_corruption', 'employment_agriculture', 'gini']
Targets:  ['poverty_3', 'poverty_8_30', 'poverty_10']


## Load data

In [2]:
train_full = pd.read_csv(DATA_PROCESSED / "train.csv")
test_full = pd.read_csv(DATA_PROCESSED / "test.csv")

print(f"Train: {train_full.shape}")
print(f"Test:  {test_full.shape}")

Train: (1626, 14)
Test:  (373, 14)


## Define model configurations

No hyperparameter tuning (grid search / cross-validation) — we use reasonable defaults informed by common practice. This is deliberate: identical conditions ensure a fair model-family comparison.

In [3]:
def get_models():
    """Return a dict of model name -> model instance (fresh for each target)."""
    models = {}

    if HAS_XGBOOST:
        # More trees + lower LR is standard practice for better generalisation
        models["XGBoost_CPU"] = XGBRegressor(
            n_estimators=2000, max_depth=15, learning_rate=0.01,
            subsample=0.95, colsample_bytree=0.95,
            reg_alpha=0.0, reg_lambda=0.1, min_child_weight=1,
            tree_method="hist", device="cpu", random_state=42, verbosity=0
        )
        models["XGBoost_GPU"] = XGBRegressor(
            n_estimators=2000, max_depth=15, learning_rate=0.01,
            subsample=0.95, colsample_bytree=0.95,
            reg_alpha=0.0, reg_lambda=0.1, min_child_weight=1,
            tree_method="hist", device="cuda", random_state=42, verbosity=0
        )
    else:
        print("  Skipping XGBoost models (library not available)")

    if HAS_LIGHTGBM:
        models["LightGBM"] = LGBMRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=42, verbosity=-1
        )
    else:
        print("  Skipping LightGBM model (library not available)")

    # RF uses deeper trees (max_depth=15) because individual trees are not boosted
    models["RandomForest"] = RandomForestRegressor(
        n_estimators=300, max_depth=15, random_state=42, n_jobs=-1
    )
    models["Ridge"] = Ridge(alpha=1.0)
    models["MLP"] = MLPRegressor(
        hidden_layer_sizes=(32, 16), max_iter=500, random_state=42,
        early_stopping=True, validation_fraction=0.15
    )

    return models

# Models that need scaled features
NEEDS_SCALING = {"Ridge", "MLP"}

print(f"Models: {list(get_models().keys())} + GAM")
print(f"Need scaling: {NEEDS_SCALING}")

Models: ['XGBoost_CPU', 'XGBoost_GPU', 'LightGBM', 'RandomForest', 'Ridge', 'MLP'] + GAM
Need scaling: {'Ridge', 'MLP'}


## Training loop

For each poverty threshold × each model:
1. Filter rows where the target is not NaN
2. Scale features for Ridge/MLP (fit scaler on train only)
3. Fit the model
4. Save to `models/{model_name}_{threshold}.pkl`
5. For Ridge/MLP, also save the scaler

In [4]:
# Track training times
timing_records = []

for target in TARGETS:
    print("=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)
    
    # Filter rows where this target is not NaN
    train_df = train_full.dropna(subset=[target])
    test_df = test_full.dropna(subset=[target])
    
    X_train = train_df[FEATURES].values
    y_train = train_df[target].values
    X_test = test_df[FEATURES].values
    y_test = test_df[target].values
    
    print(f"  Train rows: {len(X_train)}, Test rows: {len(X_test)}")
    
    # Fit scaler on train data (for Ridge/MLP)
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler for this target
    joblib.dump(scaler, MODELS_DIR / f"scaler_{target}.pkl")
    
    # --- Train sklearn/xgboost/lightgbm models ---
    models = get_models()
    for model_name, model in models.items():
        
        # Choose scaled or unscaled features
        X_tr = X_train_scaled if model_name in NEEDS_SCALING else X_train
        
        # GPU fallback for XGBoost
        if model_name == "XGBoost_GPU":
            try:
                t0 = time.time()
                model.fit(X_tr, y_train)
                elapsed = time.time() - t0
            except Exception as e:
                print(f"  {model_name}: GPU failed ({e}), falling back to CPU")
                model = XGBRegressor(
                    n_estimators=300, max_depth=6, learning_rate=0.1,
                    tree_method="hist", device="cpu", random_state=42, verbosity=0
                )
                t0 = time.time()
                model.fit(X_tr, y_train)
                elapsed = time.time() - t0
        else:
            t0 = time.time()
            model.fit(X_tr, y_train)
            elapsed = time.time() - t0
        
        # Save model
        save_path = MODELS_DIR / f"{model_name}_{target}.pkl"
        joblib.dump(model, save_path)
        
        timing_records.append({
            "target": target, "model": model_name, "time_sec": round(elapsed, 3)
        })
        print(f"  {model_name:20s} — {elapsed:.3f}s — saved to {save_path.name}")
    
    # --- Train GAM separately (different API) ---
    # GAM with splines does not require feature scaling
    t0 = time.time()
    gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4))
    gam.fit(X_train, y_train)
    elapsed = time.time() - t0
    
    save_path = MODELS_DIR / f"GAM_{target}.pkl"
    joblib.dump(gam, save_path)
    
    timing_records.append({
        "target": target, "model": "GAM", "time_sec": round(elapsed, 3)
    })
    print(f"  {'GAM':20s} — {elapsed:.3f}s — saved to {save_path.name}")
    print()

TARGET: poverty_3
  Train rows: 1626, Test rows: 373
  XGBoost_CPU          — 8.532s — saved to XGBoost_CPU_poverty_3.pkl
  XGBoost_GPU          — 8.744s — saved to XGBoost_GPU_poverty_3.pkl
  LightGBM             — 0.594s — saved to LightGBM_poverty_3.pkl
  RandomForest         — 0.243s — saved to RandomForest_poverty_3.pkl
  Ridge                — 0.013s — saved to Ridge_poverty_3.pkl
  MLP                  — 0.645s — saved to MLP_poverty_3.pkl
  GAM                  — 0.083s — saved to GAM_poverty_3.pkl

TARGET: poverty_8_30
  Train rows: 1527, Test rows: 338
  XGBoost_CPU          — 9.160s — saved to XGBoost_CPU_poverty_8_30.pkl
  XGBoost_GPU          — 9.435s — saved to XGBoost_GPU_poverty_8_30.pkl
  LightGBM             — 0.725s — saved to LightGBM_poverty_8_30.pkl
  RandomForest         — 0.482s — saved to RandomForest_poverty_8_30.pkl
  Ridge                — 0.002s — saved to Ridge_poverty_8_30.pkl
  MLP                  — 0.771s — saved to MLP_poverty_8_30.pkl
  GAM          

## Training time summary

In [5]:
timing_df = pd.DataFrame(timing_records)

# Pivot: rows = model, columns = target
timing_pivot = timing_df.pivot(index="model", columns="target", values="time_sec")
timing_pivot["total"] = timing_pivot.sum(axis=1)
timing_pivot = timing_pivot.sort_values("total")

print("Training time (seconds):")
print(timing_pivot.round(3))
print(f"\nTotal training time: {timing_df['time_sec'].sum():.1f}s")

Training time (seconds):
target        poverty_10  poverty_3  poverty_8_30   total
model                                                    
Ridge              0.002      0.013         0.002   0.017
GAM                0.046      0.083         0.066   0.195
RandomForest       0.244      0.243         0.482   0.969
MLP                0.519      0.645         0.771   1.935
LightGBM           0.703      0.594         0.725   2.022
XGBoost_GPU        9.777      8.744         9.435  27.956
XGBoost_CPU       11.667      8.532         9.160  29.359

Total training time: 62.5s


## Verify saved models

In [6]:
saved_models = sorted(MODELS_DIR.glob("*.pkl"))
print(f"Total files saved: {len(saved_models)}")
for p in saved_models:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:45s} {size_kb:8.1f} KB")

Total files saved: 24
  GAM_poverty_10.pkl                                85.0 KB
  GAM_poverty_3.pkl                                 85.0 KB
  GAM_poverty_8_30.pkl                              85.0 KB
  LightGBM_poverty_10.pkl                          491.4 KB
  LightGBM_poverty_3.pkl                           394.9 KB
  LightGBM_poverty_8_30.pkl                        486.9 KB
  MLP_poverty_10.pkl                                33.1 KB
  MLP_poverty_3.pkl                                 34.5 KB
  MLP_poverty_8_30.pkl                              35.5 KB
  RandomForest_poverty_10.pkl                    34983.2 KB
  RandomForest_poverty_3.pkl                     29120.7 KB
  RandomForest_poverty_8_30.pkl                  33654.8 KB
  Ridge_poverty_10.pkl                               0.6 KB
  Ridge_poverty_3.pkl                                0.6 KB
  Ridge_poverty_8_30.pkl                             0.6 KB
  XGBoost_CPU_poverty_10.pkl                     49342.4 KB
  XGBoost_CPU_pove

---
**Outputs produced:** 21 model files + 3 scaler files in `models/`  
- 7 models × 3 poverty thresholds = 21 `.pkl` files  
- 3 `scaler_{target}.pkl` files (for Ridge/MLP at prediction time)  